In [1]:
import pandas as pd
import os
import ffmpeg
from transformers import pipeline
import yt_dlp
import torch
import tempfile
from tqdm import tqdm
import gc

/home/jovyan/.conda-envs/Marc_Qwen/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_videos=pd.read_csv("df_comments_final_v2.csv")
df_videos=df_videos.rename(columns={"video_published_at":"Date", "video_url":"Link"})
df_videos=df_videos[['Date', "Link"]]
df_videos['Date']=pd.to_datetime(df_videos["Date"], utc=True).dt.strftime("%Y-%m-%d")
df_videos=df_videos.drop_duplicates("Link")
df_videos.reset_index(drop=True)


,Date,Link
0,2023-09-12,https://www.youtube.com/watch?v=aYmdZWwr7gs
1,2023-09-02,https://www.youtube.com/watch?v=F8KUdvmW2EQ
2,2023-09-06,https://www.youtube.com/watch?v=Fd1wpNkv-Nw
3,2023-10-27,https://www.youtube.com/watch?v=AKJwinM__nU
4,2023-10-14,https://www.youtube.com/watch?v=KuVHq39boj0
...,...,...
70,2025-08-29,https://www.youtube.com/watch?v=Tq7P83VwBM0
71,2025-08-09,https://www.youtube.com/watch?v=jjSCy00AS7o
72,2025-09-28,https://www.youtube.com/watch?v=H38O5u4GI5w
73,2025-09-02,https://www.youtube.com/watch?v=AnunTXBwOlk


In [3]:
# initialising ROBERTA 
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from langdetect import detect

#declaring models for english
tokenizer_en=AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
model_en = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")

# Multilingual model (XLM-RoBERTa)
tokenizer_multi = AutoTokenizer.from_pretrained("cardiffnlp/twitter-xlm-roberta-base-sentiment", use_fast=False)
model_multi = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-xlm-roberta-base-sentiment")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_en.to(device)
model_multi.to(device)


#first function for detecting language
def detect_lang(text):
    try:
        return detect(text)
    except:
        return "unkown"
    
#creating label map:
label_map={0:'negative', 1:'neutral', 2:'positive'}
#second function for getting sentiment

def get_sentiment(lang,text):
    if lang=='en':
        tokenizer=tokenizer_en
        model=model_en
    else:
        tokenizer=tokenizer_multi
        model=model_multi
    tokenized_text=tokenizer(text, return_tensors="pt", add_special_tokens=False)
    tokenized_text = {k: v.to(device) for k, v in tokenized_text.items()}
    model.eval()
    input_ids = tokenized_text["input_ids"][0].to(device)

    # Chunk size: 510 to leave room for <s> and </s>
    chunk_size = 510
    chunks = [input_ids[i:i+chunk_size] for i in range(0, len(input_ids), chunk_size)]

    input_tensors = [
        torch.cat([
            torch.tensor([tokenizer.cls_token_id], device=device),
            chunk,
            torch.tensor([tokenizer.sep_token_id], device=device)
        ])
        for chunk in chunks
        ]


    # Pad sequences to same length for batch processing
    from torch.nn.utils.rnn import pad_sequence
    input_batch = pad_sequence(input_tensors, batch_first=True, padding_value=tokenizer.pad_token_id)
    # Attention masks
    # long converts the mask from true/false to integers (0/1)
    #tokenizer.pad_token_id is equivalent to 1 for this type of tokenizer
    
    attention_masks = (input_batch != tokenizer.pad_token_id).long()

    with torch.no_grad():
        outputs=model(input_ids=input_batch, attention_mask=attention_masks)
        logits=outputs.logits
        probs=torch.softmax(logits, dim=-1)
    
# Aggregate probabilities across chunks
    average_probs = probs.mean(dim=0)  # shape: [num_classes]

    # Final predicted class
    predicted_class = torch.argmax(average_probs).item()  # single number
    confidence = average_probs[predicted_class].item()     # confidence of final prediction

    # Final sentiment prediction
    predicted_class = torch.argmax(average_probs).item()

    return{
        "label":label_map[predicted_class],
        "confidence":confidence,
        'probabilities':probs.tolist(),
        'chunks':chunks,
        'average_probs':average_probs

    }

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 24358.01it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 23226.58it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
#function that gets the actual sentiment from a tensor composed of the 3 probabilities (negative, neutral, positive)
def get_score(avg_probs):
    score=avg_probs[2].item()-avg_probs[0].item()
    return score


In [5]:
#loading ASR - Automatic Speech Recognition

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if torch.cuda.is_available() else -1
)

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 6750.55it/s]


In [6]:
def get_audio_file(url):
    
    temp_dir = tempfile.mkdtemp()
    output_template = os.path.join(temp_dir, "%(id)s.%(ext)s")

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": output_template,
        "noplaylist": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        file_path = ydl.prepare_filename(info)

    return file_path

In [7]:
def get_text(file_path):
    #transcribe text, enabling VAD (Voice Activity Detection)
    text_from_audio=asr(file_path,return_timestamps=True, generate_kwargs={
        "no_speech_threshold": 0.6,          # skip silence/noise
        "logprob_threshold": -1.0,           # reject low-confidence tokens
        "compression_ratio_threshold": 2.4,  # avoid repetitiions
        "temperature": 0.2                   # more conservative decoding
    })
    # Clean up temp file
    os.remove(file_path)
    return text_from_audio['text']

In [ ]:
from tqdm import tqdm
import gc
# Ensure index is continuous
df_videos = df_videos.reset_index(drop=True)


for i in tqdm(range(len(df_videos))):

    url=df_videos.loc[i, 'Link']
    print(f"Processing video {i+1}")
    
    try:
        path = get_audio_file(url)
        if not path:
            continue
    
        text = get_text(path)
        if not text:
            continue
    
        df_videos.at[i, 'video_text'] = text
        lang = detect_lang(text)
        sentiment = get_sentiment(lang, text)
    
        score = float(get_score(sentiment['average_probs']))
        df_videos.at[i, 'sentiment_tensor'] = score
    
    except Exception as e:
        print(f"Error at index {i}: {e}")
        continue

    

# clean memory
gc.collect()
torch.cuda.empty_cache()

  0%|          | 0/75 [00:00<?, ?it/s]

Processing video 1
[youtube] Extracting URL: https://www.youtube.com/watch?v=aYmdZWwr7gs
[youtube] aYmdZWwr7gs: Downloading webpage


[youtube] aYmdZWwr7gs: Downloading android vr player API JSON
[info] aYmdZWwr7gs: Downloading 1 format(s): 251
[download] Destination: /tmp/tmp__sxlr1y/aYmdZWwr7gs.webm
[download] 100% of   27.24MiB in 00:00:03 at 7.16MiB/s     


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

Processing video 2
[youtube] Extracting URL: https://www.youtube.com/watch?v=F8KUdvmW2EQ
[youtube] F8KUdvmW2EQ: Downloading webpage


...

Processing video 56
[youtube] Extracting URL: https://www.youtube.com/watch?v=oiKYco-DLTM
[youtube] oiKYco-DLTM: Downloading webpage


[youtube] oiKYco-DLTM: Downloading android vr player API JSON
[info] oiKYco-DLTM: Downloading 1 format(s): 251
[download] Destination: /tmp/tmpzx9civaw/oiKYco-DLTM.webm
[download] 100% of   14.61MiB in 00:00:03 at 4.40MiB/s     


 75%|███████▍  | 56/75 [45:29<15:52, 50.15s/it]

Processing video 57
[youtube] Extracting URL: https://www.youtube.com/watch?v=QGJysv_Qzkw
[youtube] QGJysv_Qzkw: Downloading webpage


[youtube] QGJysv_Qzkw: Downloading android vr player API JSON
[info] QGJysv_Qzkw: Downloading 1 format(s): 251
[download] Destination: /tmp/tmpyz6mtsv3/QGJysv_Qzkw.webm
[download]  59.4% of   50.07MiB at   27.10MiB/s ETA 00:00  

In [ ]:
df_videos.to_csv("df_videos_text_and_sentiment.csv")